# An Adapted Learned Policy

## Separating training from independent evaluation

The learned position at date $k$ is written as

$$
\phi_k=f_{\theta}\!\left(T-t_k,\log(S_k/K),\phi_{k-1}\right).
$$

Here, $f_{\theta}$ is the released shared policy, $\theta$ contains its fitted parameters, $T-t_k$ is remaining time, $S_k$ is the current price, $K$ is the strike, and $\phi_{k-1}$ is the previous holding. No future price enters this mapping.

### Three Declared Training Seeds

In [1]:
from pathlib import Path
import hashlib
import sys
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display as _display


def display(value):
    module = value.__class__.__module__
    if module.startswith("pandas") and hasattr(value, "_repr_html_"):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + "</div>"))
    return _display(value)


BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
FINANCIAL_ROOT = BOOK_ROOT

import RiskLabAI
from RiskLabAI.backtest.deep_hedging import DeepHedge, black_scholes_call_delta, entropic_risk, gbm_hedging_paths, hedging_pnl, train_deep_hedge

assert RiskLabAI.__version__ == "3.2.0"

torch.set_num_threads(1)
torch.set_default_dtype(torch.float64)

FIGURES = BOOK_ROOT / "outputs" / "figures" / "31_deep_hedging"
TABLES = BOOK_ROOT / "outputs" / "tables" / "31_deep_hedging"
FIGURE_DATA = BOOK_ROOT / "outputs" / "figure_data" / "31_deep_hedging"
ARTIFACTS = BOOK_ROOT / "outputs" / "artifacts" / "31_deep_hedging"
DATA_FILE = ensure_dataset(BOOK_ROOT, "fred_nasdaq")
for directory in [FIGURES, TABLES, FIGURE_DATA, ARTIFACTS]:
    directory.mkdir(parents=True, exist_ok=True)

PALETTE = {
    "background": "#FFFFFF",
    "axes": "#FFFFFF",
    "text": "#1B365D",
    "cyan": "#0072B2",
    "amber": "#C76D00",
    "green": "#00866A",
    "rose": "#B83B3B",
    "violet": "#A03A78",
    "grid": "#D9DEE5",
}
plt.rcParams.update({
    "font.family": "DejaVu Serif",
    "font.size": 11.5,
    "figure.facecolor": PALETTE["background"],
    "axes.facecolor": PALETTE["axes"],
    "axes.edgecolor": PALETTE["text"],
    "axes.labelcolor": PALETTE["text"],
    "axes.titlecolor": PALETTE["text"],
    "xtick.color": PALETTE["text"],
    "ytick.color": PALETTE["text"],
    "text.color": PALETTE["text"],
    "grid.color": PALETTE["grid"],
    "grid.linestyle": ":",
    "legend.facecolor": PALETTE["axes"],
    "legend.edgecolor": PALETTE["grid"],
    "savefig.facecolor": PALETTE["background"],
    "savefig.transparent": False,
    "savefig.dpi": 300,
})


def rendered_value(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (bool, np.bool_)):
        return "Pass" if bool(value) else "Fail"
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        value = float(value)
        if value == 0:
            return "0"
        if value.is_integer():
            return f"{int(value):,}"
        if abs(value) < 0.0001 or abs(value) >= 1000000:
            return f"{value:.6g}"
        return f"{value:.6f}".rstrip("0").rstrip(".")
    return str(value)


def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))


def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position="htbp")
    if fit:
        latex = latex.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def save_split_table(frame, groups, stem, caption, label):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f"{stem}.csv", index=False)
    panels = []
    for heading, columns in groups:
        tabular = shown[columns].to_latex(index=False, escape=True)
        tabular = tabular.replace("\\begin{tabular}", "\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}\\end{adjustbox}", 1)
        panels.append(f"\\textit{{{heading}}}\\par\\smallskip\n{tabular}")
    latex = f"\\begin{{table}}[htbp]\n\\caption{{{caption}}}\n\\label{{{label}}}\n\\centering\n\\small\n" + "\\medskip\n".join(panels) + "\\end{table}\n"
    (TABLES / f"{stem}.tex").write_text(latex, encoding="utf-8")
    display(shown)
    return shown


def finish_axis(axis, grid=True):
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)


def save_figure(figure, stem):
    figure.savefig(FIGURES / f"{stem}.png", dpi=300, bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    figure.savefig(FIGURES / f"{stem}.pdf", bbox_inches="tight", facecolor=PALETTE["background"], transparent=False)
    display(figure)
    plt.close(figure)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def public_volatility():
    frame = pd.read_csv(DATA_FILE, parse_dates=["observation_date"])
    frame["NASDAQCOM"] = pd.to_numeric(frame["NASDAQCOM"], errors="coerce")
    design = frame.loc[(frame["observation_date"] >= "1994-01-04") & (frame["observation_date"] <= "2014-12-31")].copy()
    observed = design.dropna(subset=["NASDAQCOM"]).copy()
    observed["Log return"] = np.log(observed["NASDAQCOM"] / observed["NASDAQCOM"].shift())
    returns = observed["Log return"].dropna()
    return frame, design, observed, returns, float(returns.std(ddof=1) * np.sqrt(252))


def cashflow_components(prices, positions, strike, cost_rate, premium=0.0):
    zero = torch.zeros_like(positions[:, :1])
    trades = torch.diff(torch.cat([zero, positions, zero], dim=1), dim=1)
    gains = (positions * torch.diff(prices, dim=1)).sum(dim=1)
    turnover = (prices * trades.abs()).sum(dim=1)
    costs = cost_rate * turnover
    payoff = torch.relu(prices[:, -1] - strike)
    pnl = premium + gains - costs - payoff
    return {"trades": trades, "gains": gains, "turnover": turnover, "costs": costs, "payoff": payoff, "pnl": pnl}


def metric_record(name, prices, positions, strike, cost_rate, risk_aversion):
    components = cashflow_components(prices, positions, strike, cost_rate)
    pnl = components["pnl"].detach()
    cutoff = torch.quantile(pnl, 0.05)
    return {
        "Strategy": name,
        "Entropic risk": entropic_risk(pnl, risk_aversion).item(),
        "Mean cash flow": pnl.mean().item(),
        "Cash flow standard deviation": pnl.std(unbiased=True).item(),
        "Fifth percentile": cutoff.item(),
        "Mean loss in worst five percent": -pnl[pnl <= cutoff].mean().item(),
        "Mean turnover": components["turnover"].mean().item(),
        "Mean trading cost": components["costs"].mean().item(),
        "Mean payoff": components["payoff"].mean().item(),
    }

full_public, public_design, public_observed, public_returns, sigma = public_volatility()
design = {
    "spot": 1.0,
    "strike": 1.0,
    "maturity_days": 30,
    "maturity": 30 / 365,
    "volatility": sigma,
    "drift": 0.0,
    "steps": 20,
    "cost_rate": 0.002,
    "risk_aversion": 3.0,
    "hidden_size": 16,
    "learning_rate": 0.003,
    "training_paths": 2048,
    "evaluation_paths": 8192,
    "iterations": 200,
}
training_seeds = [3301, 3302, 3303]
evaluation_seed = 33999
evaluation_prices = gbm_hedging_paths(spot=design["spot"], maturity=design["maturity"], volatility=design["volatility"], drift=design["drift"], steps=design["steps"], paths=design["evaluation_paths"], seed=evaluation_seed)

training_contract = pd.DataFrame([
    {"Item": "Training paths per seed", "Value": design["training_paths"]},
    {"Item": "Independent evaluation paths", "Value": design["evaluation_paths"]},
    {"Item": "Training iterations", "Value": design["iterations"]},
    {"Item": "Hidden units per layer", "Value": design["hidden_size"]},
    {"Item": "Learning rate", "Value": design["learning_rate"]},
    {"Item": "Declared training seeds", "Value": ", ".join(str(seed) for seed in training_seeds)},
    {"Item": "Evaluation seed", "Value": evaluation_seed},
    {"Item": "Evaluation used for fitting or stopping", "Value": "No"},
])
save_table(training_contract, "deep_hedging_training_contract", "Fixed training and evaluation contract.", "tab:chapter33-training-contract", fit=True)

histories = []
run_rows = []
for seed in training_seeds:
    path_seed = seed * 10 + 1
    model_seed = seed * 10 + 2
    training_prices = gbm_hedging_paths(spot=design["spot"], maturity=design["maturity"], volatility=design["volatility"], drift=design["drift"], steps=design["steps"], paths=design["training_paths"], seed=path_seed)
    model = DeepHedge(design["hidden_size"], seed=model_seed)
    started = time.perf_counter()
    history = train_deep_hedge(model, training_prices, strike=design["strike"], maturity=design["maturity"], cost_rate=design["cost_rate"], risk_aversion=design["risk_aversion"], iterations=design["iterations"], learning_rate=design["learning_rate"])
    elapsed = time.perf_counter() - started
    assert np.isfinite(history).all()
    positions = model(evaluation_prices, strike=design["strike"], maturity=design["maturity"]).detach()
    evaluation_pnl = hedging_pnl(evaluation_prices, positions, strike=design["strike"], cost_rate=design["cost_rate"])
    evaluation_risk = entropic_risk(evaluation_pnl, design["risk_aversion"]).item()
    torch.save(model.state_dict(), ARTIFACTS / f"deep_hedge_seed_{seed}.pt", _use_new_zipfile_serialization=False)
    histories.extend({"Iteration": index + 1, "Seed": seed, "Training risk": float(value)} for index, value in enumerate(history))
    run_rows.append({
        "Seed": seed,
        "Training path seed": path_seed,
        "Policy seed": model_seed,
        "Initial training risk": history[0],
        "Final training risk": history[-1],
        "Smallest training risk": history.min(),
        "Independent evaluation risk": evaluation_risk,
        "Training seconds": elapsed,
    })
history_frame = pd.DataFrame(histories)
run_summary = pd.DataFrame(run_rows)
history_frame.to_csv(ARTIFACTS / "primary_training_histories.csv", index=False)
run_summary.to_csv(ARTIFACTS / "primary_training_runs.csv", index=False)
save_split_table(run_summary, [("Seeds and initial risk", ["Seed", "Training path seed", "Policy seed", "Initial training risk"]), ("Training and independent evaluation", ["Seed", "Final training risk", "Smallest training risk", "Independent evaluation risk", "Training seconds"])], "deep_hedging_training_run_summary", "All declared primary training runs and independent evaluation risks.", "tab:chapter33-training-runs")

checkpoints = history_frame.loc[history_frame["Iteration"].isin([1, 10, 25, 50, 100, 150, 200])].copy()
save_table(checkpoints, "deep_hedging_learning_curve_checkpoints", "Training-risk checkpoints for every declared seed.", "tab:chapter33-learning-checkpoints")

history_frame.to_csv(FIGURE_DATA / "deep_hedging_training_curves.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
for seed, color in zip(training_seeds, [PALETTE["cyan"], PALETTE["amber"], PALETTE["green"]]):
    selected = history_frame.loc[history_frame["Seed"] == seed]
    axis.plot(selected["Iteration"], selected["Training risk"], label=f"Seed {seed}", color=color, linewidth=1.5)
axis.set(title="Training Risk for Every Declared Seed", xlabel="Training iteration", ylabel="Entropic risk")
axis.legend()
finish_axis(axis)
save_figure(figure, "deep_hedging_training_curves")

pivot = history_frame.pivot(index="Iteration", columns="Seed", values="Training risk")
curve_band = pd.DataFrame({"Iteration": pivot.index, "Mean": pivot.mean(axis=1), "Minimum": pivot.min(axis=1), "Maximum": pivot.max(axis=1)})
curve_band.to_csv(FIGURE_DATA / "deep_hedging_training_curve_band.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
axis.fill_between(curve_band["Iteration"], curve_band["Minimum"], curve_band["Maximum"], color=PALETTE["violet"], alpha=0.25, label="Seed range")
axis.plot(curve_band["Iteration"], curve_band["Mean"], color=PALETTE["cyan"], linewidth=2.0, label="Seed mean")
axis.set(title="Mean and Range of the Three Training Curves", xlabel="Training iteration", ylabel="Entropic risk")
axis.legend()
finish_axis(axis)
save_figure(figure, "deep_hedging_training_curve_band")

,Item,Value
0,Training paths per seed,"2,048"
1,Independent evaluation paths,"8,192"
2,Training iterations,200
3,Hidden units per layer,16
4,Learning rate,0.003
5,Declared training seeds,"3301, 3302, 3303"
6,Evaluation seed,"33,999"
7,Evaluation used for fitting or stopping,No


,Seed,Training path seed,Policy seed,Initial training risk,Final training risk,Smallest training risk,Independent evaluation risk,Training seconds
0,"3,301","33,011","33,012",0.03239,0.031695,0.031694,0.031404,2.898156
1,"3,302","33,021","33,022",0.03345,0.032465,0.032464,0.031523,1.854411
2,"3,303","33,031","33,032",0.032855,0.032468,0.032467,0.031436,1.842604


,Iteration,Seed,Training risk
0,1,"3,301",0.03239
9,10,"3,301",0.031742
24,25,"3,301",0.031715
49,50,"3,301",0.031697
99,100,"3,301",0.031695
149,150,"3,301",0.031694
199,200,"3,301",0.031695
200,1,"3,302",0.03345
209,10,"3,302",0.032544
224,25,"3,302",0.032476


<Figure size 900x520 with 1 Axes>

<Figure size 900x520 with 1 Axes>

## Enforcing the information boundary in fitted positions

Adaptation requires

$$
\phi_k\in\mathcal{F}_k,
$$

where $\mathcal{F}_k$ contains information available through date $k$. The condition says that the holding $\phi_k$ may depend on current and earlier observations, but it cannot depend on $S_{k+1},\ldots,S_N$. The numerical check changes only future prices and verifies that earlier positions remain identical.

### Verifying causal dependence through gradients

In [2]:
model = DeepHedge(design["hidden_size"], seed=training_seeds[0] * 10 + 2)
model.load_state_dict(torch.load(ARTIFACTS / f"deep_hedge_seed_{training_seeds[0]}.pt", weights_only=True))
base = evaluation_prices[:128].clone()
changed = base.clone()
cut = 10
changed[:, cut + 1:] *= 1.25
base_positions = model(base, strike=design["strike"], maturity=design["maturity"]).detach()
changed_positions = model(changed, strike=design["strike"], maturity=design["maturity"]).detach()
early_difference = torch.max(torch.abs(base_positions[:, :cut + 1] - changed_positions[:, :cut + 1])).item()
later_difference = torch.max(torch.abs(base_positions[:, cut + 1:] - changed_positions[:, cut + 1:])).item()
adaptation = pd.DataFrame([
    {"Check": "Largest earlier-position difference", "Value": early_difference, "Pass": early_difference == 0},
    {"Check": "Largest later-position difference", "Value": later_difference, "Pass": later_difference > 0},
    {"Check": "Future prices changed only after date", "Value": cut, "Pass": bool(torch.equal(base[:, :cut + 1], changed[:, :cut + 1]))},
])
save_table(adaptation, "deep_hedging_adaptation_checks", "Changing future prices leaves earlier fitted positions unchanged.", "tab:chapter33-adaptation-checks", fit=True)

training_prices_check = gbm_hedging_paths(spot=design["spot"], maturity=design["maturity"], volatility=design["volatility"], drift=design["drift"], steps=design["steps"], paths=design["evaluation_paths"], seed=training_seeds[0] * 10 + 1)
train_hash = hashlib.sha256(training_prices_check.numpy().tobytes()).hexdigest().upper()
evaluation_hash = hashlib.sha256(evaluation_prices.numpy().tobytes()).hexdigest().upper()
independence = pd.DataFrame([
    {"Check": "Training and evaluation seeds differ", "Value": f"{training_seeds[0] * 10 + 1} and {evaluation_seed}", "Pass": training_seeds[0] * 10 + 1 != evaluation_seed},
    {"Check": "Training and evaluation arrays differ", "Value": train_hash != evaluation_hash, "Pass": train_hash != evaluation_hash},
    {"Check": "Evaluation paths used after fitting", "Value": "Yes", "Pass": True},
    {"Check": "Evaluation result selected stopping time", "Value": "No", "Pass": True},
])
save_table(independence, "deep_hedging_independence_checks", "Training and final evaluation remain separate.", "tab:chapter33-independence", fit=True)

model.zero_grad()
gradient_positions = model(base, strike=design["strike"], maturity=design["maturity"])
gradient_pnl = hedging_pnl(base, gradient_positions, strike=design["strike"], cost_rate=design["cost_rate"])
gradient_risk = entropic_risk(gradient_pnl, design["risk_aversion"])
gradient_risk.backward()
gradient_values = torch.cat([parameter.grad.reshape(-1) for parameter in model.parameters() if parameter.grad is not None])
gradient_checks = pd.DataFrame([
    {"Check": "Risk is finite", "Value": gradient_risk.item(), "Pass": bool(torch.isfinite(gradient_risk))},
    {"Check": "All gradients are finite", "Value": bool(torch.isfinite(gradient_values).all()), "Pass": bool(torch.isfinite(gradient_values).all())},
    {"Check": "Largest absolute gradient", "Value": gradient_values.abs().max().item(), "Pass": bool(gradient_values.abs().max() > 0)},
])
save_table(gradient_checks, "deep_hedging_gradient_checks", "Finite risk and gradient checks after primary training.", "tab:chapter33-gradient-checks", fit=True)

all_positions = model(evaluation_prices, strike=design["strike"], maturity=design["maturity"]).detach().numpy()
position_rows = []
surface_rows = []
for index in range(design["steps"]):
    values = all_positions[:, index]
    position_rows.append({"Trading date": index, "Fifth percentile": np.quantile(values, 0.05), "First quartile": np.quantile(values, 0.25), "Median": np.median(values), "Third quartile": np.quantile(values, 0.75), "Ninety-fifth percentile": np.quantile(values, 0.95)})
    if index in [0, 5, 10, 15, 19]:
        prices_now = evaluation_prices[:, index].numpy()
        bins = pd.qcut(prices_now, q=12, duplicates="drop")
        grouped = pd.DataFrame({"Price": prices_now, "Holding": values, "Bin": bins}).groupby("Bin", observed=True).agg(Current_price=("Price", "mean"), Mean_holding=("Holding", "mean")).reset_index(drop=True)
        for row in grouped.to_dict("records"):
            surface_rows.append({"Trading date": index, "Current price": row["Current_price"], "Mean holding": row["Mean_holding"]})
position_summary = pd.DataFrame(position_rows)
save_table(position_summary.loc[position_summary["Trading date"].isin([0, 5, 10, 15, 19])], "deep_hedging_position_summary", "Selected fitted-position summaries over the trading horizon.", "tab:chapter33-position-summary", fit=True)

surface = pd.DataFrame(surface_rows)
surface.to_csv(FIGURE_DATA / "deep_hedging_position_surface.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
for index, color in zip([0, 5, 10, 15, 19], [PALETTE["text"], PALETTE["green"], PALETTE["cyan"], PALETTE["amber"], PALETTE["rose"]]):
    selected = surface.loc[surface["Trading date"] == index]
    axis.plot(selected["Current price"], selected["Mean holding"], marker="o", markersize=3, linewidth=1.4, color=color, label=f"Date {index}")
axis.set(title="Fitted Holdings by Current Price and Date", xlabel="Current normalized price", ylabel="Mean fitted holding")
axis.legend()
finish_axis(axis)
save_figure(figure, "deep_hedging_position_surface")

position_summary.to_csv(FIGURE_DATA / "deep_hedging_holding_quantiles.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
axis.fill_between(position_summary["Trading date"], position_summary["Fifth percentile"], position_summary["Ninety-fifth percentile"], color=PALETTE["violet"], alpha=0.2, label="5th to 95th percentile")
axis.fill_between(position_summary["Trading date"], position_summary["First quartile"], position_summary["Third quartile"], color=PALETTE["cyan"], alpha=0.28, label="Middle half")
axis.plot(position_summary["Trading date"], position_summary["Median"], color=PALETTE["amber"], linewidth=1.8, label="Median")
axis.set(title="Distribution of Fitted Holdings through Time", xlabel="Trading date", ylabel="Fitted holding")
axis.legend()
finish_axis(axis)
save_figure(figure, "deep_hedging_holding_quantiles")

adaptation_figure = pd.DataFrame({"Date": np.arange(design["steps"]), "Original position": base_positions[0].numpy(), "Changed-future position": changed_positions[0].numpy()})
adaptation_figure.to_csv(FIGURE_DATA / "deep_hedging_adaptation_path.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
axis.plot(adaptation_figure["Date"], adaptation_figure["Original position"], color=PALETTE["cyan"], marker="o", label="Original path")
axis.plot(adaptation_figure["Date"], adaptation_figure["Changed-future position"], color=PALETTE["rose"], marker="s", markersize=3, label="Future changed")
axis.axvline(cut, color=PALETTE["amber"], linewidth=1.3, label="Last unchanged price")
axis.set(title="Earlier Positions Do Not Change with Future Prices", xlabel="Trading date", ylabel="Fitted holding")
axis.legend()
finish_axis(axis)
save_figure(figure, "deep_hedging_adaptation_path")

time_table = run_summary[["Seed", "Training seconds"]].copy()
save_table(time_table, "deep_hedging_training_time", "Observed computer training time for every declared seed.", "tab:chapter33-training-time", fit=True)
time_table.to_csv(FIGURE_DATA / "deep_hedging_training_time.csv", index=False)
figure, axis = plt.subplots(figsize=(8.5, 4.8), constrained_layout=True)
axis.bar(time_table["Seed"].astype(str), time_table["Training seconds"], color=[PALETTE["cyan"], PALETTE["amber"], PALETTE["green"]])
axis.set(title="Observed Training Time by Seed", xlabel="Declared seed", ylabel="Seconds on this computer")
finish_axis(axis, grid=False)
save_figure(figure, "deep_hedging_training_time")

,Check,Value,Pass
0,Largest earlier-position difference,0,Pass
1,Largest later-position difference,0.00169,Pass
2,Future prices changed only after date,10,Pass


,Check,Value,Pass
0,Training and evaluation seeds differ,33011 and 33999,Pass
1,Training and evaluation arrays differ,Pass,Pass
2,Evaluation paths used after fitting,Yes,Pass
3,Evaluation result selected stopping time,No,Pass


,Check,Value,Pass
0,Risk is finite,0.037517,Pass
1,All gradients are finite,Pass,Pass
2,Largest absolute gradient,0.016573,Pass


,Trading date,Fifth percentile,First quartile,Median,Third quartile,Ninety-fifth percentile
0,0,0.254135,0.254135,0.254135,0.254135,0.254135
5,5,0.264653,0.26488,0.265015,0.265133,0.265269
10,10,0.264294,0.264656,0.264874,0.265056,0.265245
15,15,0.263886,0.264408,0.264709,0.264956,0.265204
19,19,0.26357,0.264204,0.264567,0.264853,0.265154


<Figure size 900x520 with 1 Axes>

<Figure size 900x520 with 1 Axes>

<Figure size 900x520 with 1 Axes>

,Seed,Training seconds
0,"3,301",2.898156
1,"3,302",1.854411
2,"3,303",1.842604


<Figure size 850x480 with 1 Axes>

## Independent Outcomes for Every Fitted Policy

For each declared seed $j$, the reported final risk is

$$
\widehat\rho_j=\frac{1}{\lambda}\log\!\left(\frac{1}{M}\sum_{i=1}^{M}e^{-\lambda P_{ij}}\right).
$$

Here, $M$ is the fixed number of independent evaluation paths and $P_{ij}$ is the terminal cash flow for evaluation path $i$ under the policy fitted with seed $j$. Every seed is retained; no evaluation result chooses a policy or changes the number of training iterations.

In [3]:
metric_rows = []
pnl_data = {}
holding_data = {"Trading date": np.arange(design["steps"])}
for seed in training_seeds:
    seed_model = DeepHedge(design["hidden_size"], seed=seed * 10 + 2)
    seed_model.load_state_dict(torch.load(ARTIFACTS / f"deep_hedge_seed_{seed}.pt", weights_only=True))
    seed_positions = seed_model(evaluation_prices, strike=design["strike"], maturity=design["maturity"]).detach()
    metric_rows.append(metric_record(f"Learned seed {seed}", evaluation_prices, seed_positions, design["strike"], design["cost_rate"], design["risk_aversion"]))
    pnl_data[f"Seed {seed}"] = hedging_pnl(evaluation_prices, seed_positions, strike=design["strike"], cost_rate=design["cost_rate"]).detach().numpy()
    holding_data[f"Seed {seed}"] = seed_positions.mean(dim=0).numpy()
seed_metrics = pd.DataFrame(metric_rows)
seed_metrics.to_csv(ARTIFACTS / "primary_evaluation_metrics.csv", index=False)
save_split_table(seed_metrics, [("Risk and cash flow distribution", ["Strategy", "Entropic risk", "Mean cash flow", "Cash flow standard deviation", "Fifth percentile"]), ("Tail, turnover, cost, and payoff", ["Strategy", "Mean loss in worst five percent", "Mean turnover", "Mean trading cost", "Mean payoff"])], "deep_hedging_learned_seed_metrics", "Independent outcomes for every declared learned policy.", "tab:chapter33-learned-seed-metrics")

seed_summary = pd.DataFrame([
    {"Quantity": "Smallest learned-policy risk", "Value": seed_metrics["Entropic risk"].min()},
    {"Quantity": "Largest learned-policy risk", "Value": seed_metrics["Entropic risk"].max()},
    {"Quantity": "Mean learned-policy risk", "Value": seed_metrics["Entropic risk"].mean()},
    {"Quantity": "Risk range", "Value": seed_metrics["Entropic risk"].max() - seed_metrics["Entropic risk"].min()},
])
save_table(seed_summary, "deep_hedging_seed_outcome_summary", "Range of retained learned-policy outcomes across the three seeds.", "tab:chapter33-seed-outcome-summary", fit=True)

pnl_frame = pd.DataFrame(pnl_data)
pnl_frame.to_csv(FIGURE_DATA / "deep_hedging_learned_seed_cashflows.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
for name, color in zip(pnl_frame.columns, [PALETTE["cyan"], PALETTE["amber"], PALETTE["green"]]):
    axis.hist(pnl_frame[name], bins=70, density=True, histtype="step", linewidth=1.5, label=name, color=color)
axis.set(title="Independent Cash Flows for Every Learned Policy", xlabel="Terminal cash flow", ylabel="Density")
axis.legend()
finish_axis(axis)
save_figure(figure, "deep_hedging_learned_seed_cashflows")

holding_frame = pd.DataFrame(holding_data)
holding_frame.to_csv(FIGURE_DATA / "deep_hedging_mean_holdings_by_seed.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
for name, color in zip(holding_frame.columns[1:], [PALETTE["cyan"], PALETTE["amber"], PALETTE["green"]]):
    axis.plot(holding_frame["Trading date"], holding_frame[name], linewidth=1.6, label=name, color=color)
axis.set(title="Mean Fitted Holding for Every Seed", xlabel="Trading date", ylabel="Mean holding")
axis.legend()
finish_axis(axis)
save_figure(figure, "deep_hedging_mean_holdings_by_seed")

,Strategy,Entropic risk,Mean cash flow,Cash flow standard deviation,Fifth percentile,Mean loss in worst five percent,Mean turnover,Mean trading cost,Mean payoff
0,Learned seed 3301,0.031404,-0.030042,0.029301,-0.094589,0.11859,0.530783,0.001062,0.028887
1,Learned seed 3302,0.031523,-0.030631,0.023963,-0.078803,0.097747,0.800167,0.0016,0.028887
2,Learned seed 3303,0.031436,-0.030395,0.025773,-0.084628,0.105965,0.692083,0.001384,0.028887


,Quantity,Value
0,Smallest learned-policy risk,0.031404
1,Largest learned-policy risk,0.031523
2,Mean learned-policy risk,0.031454
3,Risk range,0.000119


<Figure size 900x520 with 1 Axes>

<Figure size 900x520 with 1 Axes>